In [ ]:
#Install required packages
#pip install pandas scikit-learn matplotlib nltk joblib datasets

In [1]:
# Step 1: Imports & Setup
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from datasets import load_dataset
import joblib

# Make output prettier
pd.set_option('display.max_colwidth', 120)


c:\Users\hameedbf\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Step 2: Load IMDB dataset from Hugging Face Datasets
# This automatically downloads and caches it the first time you run it.
dataset = load_dataset("imdb")

# Convert to Pandas DataFrame for convenience
train_df = pd.DataFrame(dataset["train"])
test_df = pd.DataFrame(dataset["test"])

# Preview the data
train_df.head()

,text,label
0,I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first...,0
1,"""I Am Curious: Yellow"" is a risible and pretentious steaming pile. It doesn't matter what one's political views are ...",0
2,If only to avoid making this type of film in the future. This film is interesting as an experiment but tells no coge...,0
3,"This film was probably inspired by Godard's Masculin, féminin and I urge you to see that film instead.<br /><br />Th...",0
4,"Oh, brother...after hearing about this ridiculous film for umpteen years all I can think of is that old Peggy Lee so...",0


In [3]:
train_df['text'][66]

'There\'s not a drop of sunshine in "The Sunshine Boys", which makes the title of this alleged comedy Neil Simon\'s sole ironic moment. Simon, who adapted the script from his play (which goes uncredited), equates old age with irrational behavior--and, worse, clumsy, galumphing, mean-spirited irrational behavior. Walter Matthau is merciless on us playing an aged vaudeville performer talked into reuniting with former comedy partner George Burns for a television special (it\'s said they were a team for 43 years, which begs the question "how long did vaudeville last, anyway?"). Burns, who won a Supporting Oscar, has the misfortune of coming to the film some thirty minutes in, after which time Matthau has already blasted the material to hell and back. The noisier the movie gets, the less tolerable and watchable it is. Director Herbert Ross only did solid work when he wasn\'t coupled with one of Neil Simon\'s screenplays; here, Ross sets up gags like a thudding amateur, hammering away at bel

In [4]:
# Step 3: Clean and prepare text

def clean_text(s):
    s = s.lower()
    s = re.sub(r"http\S+", "", s)
    s = re.sub(r"[^a-z0-9\s']", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

# Clean text (optional: can sample subset for speed)
train_df["clean_text"] = train_df["text"].apply(clean_text)
test_df["clean_text"] = test_df["text"].apply(clean_text)

# (Optional) use a smaller subset for faster testing
train_small = train_df.sample(5000, random_state=42)
test_small = test_df.sample(2000, random_state=42)

X_train, y_train = train_small["clean_text"], train_small["label"]
X_test, y_test = test_small["clean_text"], test_small["label"]


In [5]:
train_df['clean_text'][66]

"there's not a drop of sunshine in the sunshine boys which makes the title of this alleged comedy neil simon's sole ironic moment simon who adapted the script from his play which goes uncredited equates old age with irrational behavior and worse clumsy galumphing mean spirited irrational behavior walter matthau is merciless on us playing an aged vaudeville performer talked into reuniting with former comedy partner george burns for a television special it's said they were a team for 43 years which begs the question how long did vaudeville last anyway burns who won a supporting oscar has the misfortune of coming to the film some thirty minutes in after which time matthau has already blasted the material to hell and back the noisier the movie gets the less tolerable and watchable it is director herbert ross only did solid work when he wasn't coupled with one of neil simon's screenplays here ross sets up gags like a thudding amateur hammering away at belligerent routines which fail to pay 

In [6]:
# Step 4: Vectorize text using TF-IDF

from sklearn.feature_extraction.text import TfidfVectorizer

# Create the TF-IDF vectorizer
tfidf = TfidfVectorizer(
    max_features=20000,      # keep top 20k most informative words/phrases
    ngram_range=(1,2),       # unigrams (1 word) + bigrams (2-word phrases)
    stop_words=None          # no stopword removal here
)

# Fit the vectorizer on the training text and transform both sets
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# Check matrix size
print("Train TF-IDF matrix shape:", X_train_tfidf.shape)
print("Test TF-IDF matrix shape:", X_test_tfidf.shape)


Train TF-IDF matrix shape: (5000, 20000)
Test TF-IDF matrix shape: (2000, 20000)


In [7]:
# Step 5: Train a classifier

from sklearn.linear_model import LogisticRegression

# Create model
clf = LogisticRegression(max_iter=200)

# Train the model
clf.fit(X_train_tfidf, y_train)

print("Model trained successfully!")


Model trained successfully!


In [8]:
# Step 6: Evaluate the model

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Predict on test set
y_pred = clf.predict(X_test_tfidf)

# Accuracy
acc = accuracy_score(y_test, y_pred)
print("Accuracy:", acc)

# Detailed precision, recall, and F1-score
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Accuracy: 0.854

Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.84      0.86      1040
           1       0.84      0.87      0.85       960

    accuracy                           0.85      2000
   macro avg       0.85      0.85      0.85      2000
weighted avg       0.85      0.85      0.85      2000


Confusion Matrix:
[[877 163]
 [129 831]]


In [9]:
# Step 7: Predict sentiment for new custom review

def predict_sentiment(review_text):
    # 1. Clean the text (same cleaning used for training)
    cleaned = clean_text(review_text)
    
    # 2. Convert to TF-IDF
    vec = tfidf.transform([cleaned])
    
    # 3. Predict
    pred = clf.predict(vec)[0]
    
    # 4. Convert label to readable output
    label = "Positive 😊" if pred == 1 else "Negative 😞"
    return label


# Example:
sample_review = "The movie was quite boring and too long."
print(predict_sentiment(sample_review))


Negative 😞


In [11]:
print(predict_sentiment(a))


Positive 😊


In [10]:
a = """Walking in to see Kit Kittredge: An American Girl, I was unaware the movie was spun off from a popular
line of toy dolls -- more than 14 million of them have been sold. Nor did I know about the anticipation and
excitement many little girls have for the movie, which is the fourth in the series (the previous three,
made for TV, are available on DVD).
I still have no opinion on the toys, but I can definitely vouch for the film living up to expectations. Shot
with the burnished, luxurious cinematography of a big-budget movie, and cast with actors who never
treat the material as if it were beneath them, Kit Kittredge: An American Girl is a thoroughly satisfying
and engaging children's picture that never forgets those kids probably didn't get to the theater by
themselves.
Movie Reviews
These examples were found on www.rottentomatoes.com. What I like
about this site is that it compiles many newspaper reviews onto one page.
Visitors can then make additional comments (reviewing the reviewers).
Here are a few movies that are geared towards kids…
Director Patricia Rozema and screenwriter Ann Peacock avoid condescending to their target audience --
or, for that matter, to their adult guardians -- by treating their child protagonists with care and respect.
It helps, too, that they have the amazing Abigail Breslin (Little Miss Sunshine) in the lead role of 9-yearold Kit, a bright and winsome girl in Depression-era Cincinnati who aspires to be a reporter for the
Cincinnati Register.
Kit is constantly submitting to the paper her stories, such as a piece on the World's Fair, without much
luck. But as the Depression starts to affect the lives of her closest friends -- and then, inevitably, her own
family -- Kit's priorities change. Her father (Chris O'Donnell) goes to Chicago to find work and promises to
return, but doesn't. Her mother (Julia Ormond), in a desperate attempt to save their home, starts taking
in boarders, such as a traveling librarian (Joan Cusack), a magician (Stanley Tucci) and a fallen socialite
(Glenn Headley) awaiting word from her own husband, who has gone to New York in search of
employment.
The backdrop is grim, and to its credit, the movie does not sugarcoat its portrayal of how the Depression
affected the day-to-day lives of upper middle-class Americans who lost everything in the span of a month
and had to figure out how to push forward.
The filmmakers allow the story to unfold entirely through the point of view of Kit, who may not fully
understand the severity of the situation, but is also more aware and resourceful than the adults around
her realize. Kit Kittredge: An American Girl sends its heroine and her friends on several adventures,
including a Nancy Drew-ish one involving some stolen valuables, but it's the movie's overall tone -- the
innocence of children, as well as their surprising resiliency -- that elevates the film into something far
more valuable than a feature-length commercial for toys."""

In [12]:
import pickle

# Save the TF-IDF vectorizer
with open("tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf, f)

# Save the trained model
with open("sentiment_model.pkl", "wb") as f:
    pickle.dump(clf, f)

print("Model and TF-IDF saved successfully!")


Model and TF-IDF saved successfully!
